# GeoDiff3D — GPU Proof of Concept (T4-safe)

**Phase 2A:** VGGT geometric depth + Marigold diffusion depth → robust scale/shift alignment → confidence-guided fusion → multi-view point-cloud export.

This notebook is designed for a Google Colab T4. It avoids installing/replacing Colab's preinstalled PyTorch stack, uses the current VGGT `facebook/VGGT-1B` inference API, and uses the current Diffusers Marigold `prs-eth/marigold-depth-v1-1` API. The current VGGT documentation uses `model(images)` and returns `pose_enc`, `depth`, and `depth_conf`; Marigold returns `prediction`, not `depth_np`. citeturn1search4turn0search4turn0search6

In [ ]:
# 1. HARDWARE CHECK — do this before installing anything
!nvidia-smi

import os, sys, gc, time, json, math, shutil, subprocess
import numpy as np
import torch

assert torch.cuda.is_available(), "CUDA is not available. In Colab select Runtime > Change runtime type > T4 GPU."
device = torch.device("cuda:0")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print("CUDA OK")


In [ ]:
# 2. Install only user-space dependencies.
!pip -q install "pillow<11.3"
# IMPORTANT: do NOT reinstall torch/torchvision in Colab; doing so is a common source of CUDA/runtime breakage.
!pip -q install -U "diffusers>=0.35.0" "transformers>=4.45.0" accelerate safetensors huggingface_hub matplotlib scipy plyfile

# Clone/update VGGT without touching Colab's torch installation.
!rm -rf /content/vggt
!git clone -q --depth 1 https://github.com/facebookresearch/vggt.git /content/vggt

import sys, os
if "/content/vggt" not in sys.path:
    sys.path.insert(0, "/content/vggt")

print("Dependencies installed and VGGT source available.")


In [ ]:
# 3. Imports and reproducibility
import gc, os, time, json, traceback, warnings
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("/content/geodiff3d")
INPUT_DIR = ROOT / "input"
OUTPUT_DIR = ROOT / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
print("Output:", OUTPUT_DIR)


In [ ]:
# 4. Input images
# Provide 2-12 overlapping RGB images. Either place them in /content/geodiff3d/input
# before running this cell, or use the upload widget below (individual images or a single .zip).
import io
import zipfile
from PIL import Image, UnidentifiedImageError

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def _valid_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im.convert("RGB")
        return True
    except (UnidentifiedImageError, OSError):
        return False

existing = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])

if len(existing) >= 2:
    image_paths = [p for p in existing if _valid_image(p)]
    print(f"Using {len(image_paths)} existing input images from {INPUT_DIR}.")
else:
    from google.colab import files
    print("Upload 2-12 overlapping RGB images (.jpg/.jpeg/.png/.webp), or a single .zip containing them.")
    uploaded = files.upload()
    image_paths = []
    for name, content in uploaded.items():
        suffix = Path(name).suffix.lower()
        if suffix == ".zip":
            with zipfile.ZipFile(io.BytesIO(content)) as zf:
                for member in zf.namelist():
                    m_suffix = Path(member).suffix.lower()
                    if m_suffix in SUPPORTED_EXTS and not member.endswith("/"):
                        dst = INPUT_DIR / Path(member).name
                        with open(dst, "wb") as f:
                            f.write(zf.read(member))
                        if _valid_image(dst):
                            image_paths.append(dst)
                        else:
                            dst.unlink(missing_ok=True)
                            print(f"Skipping unreadable image in zip: {member}")
        elif suffix in SUPPORTED_EXTS:
            dst = INPUT_DIR / name
            with open(dst, "wb") as f:
                f.write(content)
            if _valid_image(dst):
                image_paths.append(dst)
            else:
                dst.unlink(missing_ok=True)
                print(f"Skipping unreadable file: {name}")
        else:
            print(f"Skipping unsupported file type: {name}")

image_paths = sorted(set(image_paths))
if not (2 <= len(image_paths) <= 12):
    raise RuntimeError(f"Need between 2 and 12 valid images, got {len(image_paths)}. Re-run this cell and upload again.")

print("Inputs:")
for p in image_paths:
    with Image.open(p) as im:
        print(f"  {p.name}: {im.size}, mode={im.mode}")

fig, axes = plt.subplots(1, len(image_paths), figsize=(4*len(image_paths), 4))
axes = np.atleast_1d(axes)
for ax, p in zip(axes, image_paths):
    ax.imshow(Image.open(p).convert("RGB"))
    ax.set_title(p.name)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "input_grid.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 5. GeoDiff3D package
# This notebook is a thin client of the real geodiff3d package (core/math.py,
# inference/vggt_pipeline.py, inference/marigold_pipeline.py,
# inference/gpu_pipeline.py) -- the same code the FastAPI backend's
# VGGTMarigoldEngine calls. There is only one implementation of the math and
# the inference calls; this notebook does not redefine them.
import zipfile
from pathlib import Path
from google.colab import files

PKG_DIR = Path("/content/geodiff3d_pkg")
if not (PKG_DIR / "core" / "math.py").exists():
    print("Upload geodiff3d_pkg.zip (zip of the repo's core/ and inference/ directories).")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(PKG_DIR)

import sys
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))

from inference.gpu_pipeline import run_reconstruction
print("GeoDiff3D package loaded from", PKG_DIR)


In [ ]:
# 6. Run the real reconstruction pipeline
# One call into the actual package -- VGGT is loaded/run/freed, then Marigold
# is loaded/run/freed, then alignment, fusion, reconstruction and evaluation
# all happen inside inference/gpu_pipeline.run_reconstruction. No logic is
# duplicated here.
def _progress(state, message=None):
    print(f"[{state}]" + (f" {message}" if message else ""))

metrics = run_reconstruction(
    [str(p) for p in image_paths],
    OUTPUT_DIR,
    device=device,
    progress_cb=_progress,
)

print("\nSUCCESS")
print(f"Baseline points: {metrics['baseline']['num_points']:,}")
print(f"Guided points:   {metrics['guided']['num_points']:,}")
print(f"VGGT runtime:    {metrics['vggt']['runtime_sec']}s")
print(f"Marigold runtime:{metrics['marigold']['runtime_sec']}s")


In [ ]:
# 7. Final artifact check
for rel in [
    "visualizations/input_grid.png",
    "visualizations/depth_comparison.png",
    "metrics/alignment_metrics.json",
    "metrics/metrics.json",
    "baseline/baseline.ply",
    "guided/guided.ply",
]:
    p = OUTPUT_DIR / rel
    print(f"{'OK' if p.exists() and p.stat().st_size > 0 else 'MISSING'}  {p}  {p.stat().st_size if p.exists() else 0} bytes")

print("\nGeoDiff3D GPU POC finished without intentionally swallowing inference errors.")

from PIL import Image
display(Image.open(OUTPUT_DIR / "visualizations" / "depth_comparison.png"))
